In [2]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import os
from dotenv import load_dotenv

load_dotenv()

mongo_uri = os.getenv("MONGODB_URI")

client = MongoClient(mongo_uri, server_api=ServerApi("1"))

db = client["IBGE"]

print(db)

Database(MongoClient(host=['ac-2ggy4lg-shard-00-02.pgwfdqm.mongodb.net:27017', 'ac-2ggy4lg-shard-00-00.pgwfdqm.mongodb.net:27017', 'ac-2ggy4lg-shard-00-01.pgwfdqm.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, authsource='admin', replicaset='atlas-y9e2xz-shard-0', tls=True, server_api=<pymongo.server_api.ServerApi object at 0x000001C2EF499550>), 'IBGE')


In [3]:
data = db["PNADC"]
print(data)

Collection(Database(MongoClient(host=['ac-2ggy4lg-shard-00-02.pgwfdqm.mongodb.net:27017', 'ac-2ggy4lg-shard-00-00.pgwfdqm.mongodb.net:27017', 'ac-2ggy4lg-shard-00-01.pgwfdqm.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, authsource='admin', replicaset='atlas-y9e2xz-shard-0', tls=True, server_api=<pymongo.server_api.ServerApi object at 0x000001C2EF499550>), 'IBGE'), 'PNADC')


In [4]:
data.count_documents({})

1

In [5]:
data = list(data.find())

In [6]:
serie = data[0]["resultados"][0]["series"][0]["serie"]

In [7]:
serie

{'201201': '9.6',
 '201202': '8.3',
 '201203': '9.4',
 '201204': '9.2',
 '201301': '10.7',
 '201302': '9.7',
 '201303': '8.5',
 '201304': '7.4',
 '201401': '8.8',
 '201402': '8.0',
 '201403': '8.5',
 '201404': '7.7',
 '201501': '8.2',
 '201502': '9.2',
 '201503': '11.3',
 '201504': '11.1',
 '201601': '13.4',
 '201602': '14.2',
 '201603': '15.5',
 '201604': '15.9',
 '201701': '17.3',
 '201702': '19.0',
 '201703': '18.1',
 '201704': '17.0',
 '201801': '17.9',
 '201802': '17.1',
 '201803': '17.0',
 '201804': '15.6',
 '201901': '16.3',
 '201902': '16.1',
 '201903': '16.0',
 '201904': '14.2',
 '202001': '14.8',
 '202002': '...',
 '202003': '...',
 '202004': '...',
 '202101': '...',
 '202102': '...',
 '202103': '...',
 '202104': '...',
 '202201': '...',
 '202202': '13.6',
 '202203': '14.0',
 '202204': '12.3',
 '202301': '14.1',
 '202302': '14.2',
 '202303': '13.3',
 '202304': '12.0',
 '202401': '12.4',
 '202402': '11.6',
 '202403': '10.6',
 '202404': '10.3',
 '202501': '11.6',
 '202502': '10

In [8]:
type(serie)

dict

In [9]:
import pandas as pd

df = pd.DataFrame.from_dict(serie, orient="index", columns=["valor"])
df.index.name = "periodo"
df = df.reset_index()
df

,periodo,valor
0,201201,9.6
1,201202,8.3
2,201203,9.4
3,201204,9.2
4,201301,10.7
5,201302,9.7
6,201303,8.5
7,201304,7.4
8,201401,8.8
9,201402,8.0


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   periodo  58 non-null     str  
 1   valor    58 non-null     str  
dtypes: str(2)
memory usage: 1.0 KB


In [11]:
df["valor"] = df["valor"].replace("...", "0")
df["valor"] = df["valor"].astype(float)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   periodo  58 non-null     str    
 1   valor    58 non-null     float64
dtypes: float64(1), str(1)
memory usage: 1.0 KB


In [13]:
df["ano"] = df["periodo"].str[:4]
df["tri"] = df["periodo"].str[-2:].astype(int)

In [14]:
df

,periodo,valor,ano,tri
0,201201,9.6,2012,1
1,201202,8.3,2012,2
2,201203,9.4,2012,3
3,201204,9.2,2012,4
4,201301,10.7,2013,1
5,201302,9.7,2013,2
6,201303,8.5,2013,3
7,201304,7.4,2013,4
8,201401,8.8,2014,1
9,201402,8.0,2014,2


In [15]:
df["periodo"] = pd.PeriodIndex(df["ano"] + "Q" + df["tri"].astype(str), freq="Q")

In [16]:
df

,periodo,valor,ano,tri
0,2012Q1,9.6,2012,1
1,2012Q2,8.3,2012,2
2,2012Q3,9.4,2012,3
3,2012Q4,9.2,2012,4
4,2013Q1,10.7,2013,1
5,2013Q2,9.7,2013,2
6,2013Q3,8.5,2013,3
7,2013Q4,7.4,2013,4
8,2014Q1,8.8,2014,1
9,2014Q2,8.0,2014,2


In [17]:
df["periodo"] = df["periodo"].dt.to_timestamp()

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   periodo  58 non-null     datetime64[us]
 1   valor    58 non-null     float64       
 2   ano      58 non-null     str           
 3   tri      58 non-null     int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 1.9 KB


In [19]:
import sqlite3

conn = sqlite3.connect("ibge.db")
df.to_sql("pnadc", conn, if_exists="replace", index=False)
conn.close()